# 통합반 1회 · 센서와 데이터 흐름 알아보기 (제공 노트북)

이 노트북은 위에서 아래로 한 셀씩 실행한다. 셀을 고르고 **▶ 실행 버튼**(또는 Shift+Enter)을 누른다.

- 활동 2: 세 센서 파일(TS1·PS1·FS1)을 열어 모양을 본다
- 활동 3: 설비·센서·단위 매핑 표의 `____` 빈칸을 채운다
- 활동 4: 재생 셀로 그래프를 확인하고, 품질 플래그를 붙여 본 뒤 `saved/` 에 저장본을 남긴다

> 빈칸은 `____` 로 표시했다. 빈칸을 채우지 않아도 셀은 실행되지만, 마지막 **자기 점검** 셀이 통과하지 않는다.

In [ ]:
# 0. 준비 — lecture/system 의 제공 코드를 불러온다 (이 셀은 고치지 않는다)
import sys, logging, warnings
from pathlib import Path
from datetime import datetime, timezone

SYSTEM = next(p / 'system' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'system' / 'hydops').exists())
sys.path.insert(0, str(SYSTEM))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
plt.rcParams['font.family'] = ['Pretendard', 'AppleGothic', 'Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

from hydops.b1_data import uci

____ = None  # 빈칸 표시. 빈칸을 채우면 이 값은 쓰이지 않는다
RAW = SYSTEM / 'data' / 'raw'
print('제공 코드 위치:', SYSTEM.name, '/ 원본 폴더:', RAW.relative_to(SYSTEM))

## 활동 2 · 세 센서 파일을 연다

UCI 원본은 **탭으로 구분한 표 파일**이다. 한 줄이 한 사이클(60초)이고, 한 칸이 한 번의 측정이다.
그래서 **열의 개수 ÷ 60초 = 1초에 몇 번 쟀는지(Hz)** 가 된다.

In [ ]:
# 2-1. 원본 파일의 모양 (PS1 은 파일이 커서 앞 3줄만 읽는다)
for name in ['TS1.txt', 'PS1.txt', 'FS1.txt']:
    head = pd.read_csv(RAW / name, sep='\t', header=None, nrows=3)
    print(f'{name:8s} 한 줄(사이클)의 칸 수 = {head.shape[1]:5d}  →  {head.shape[1] // 60:3d} Hz   첫 값 = {head.iloc[0, 0]}')
with open(RAW / 'TS1.txt') as f:
    n_cycles = sum(1 for _ in f)
print('사이클(줄) 수 =', n_cycles)

In [ ]:
# 2-2. profile.txt — 사이클마다 붙은 정답 라벨 (탐지 입력에는 쓰지 않는다)
profile = uci.load_profile()
print(profile.loc[[0, 100, 1500], ['cooler_pct', 'valve_pct', 'pump_leak', 'accumulator_bar', 'stable_flag']])

In [ ]:
# 2-3. 강사 제공 축약본 — 모든 센서를 1초 단위(mean/min/max/n)로 줄여 둔 파일
ds = uci.load_reduced()
for sid, stats in ds.sensors.items():
    if sid in ('TS1', 'PS1', 'FS1'):
        print(sid, '→', {k: v.shape for k, v in stats.items()})

In [ ]:
# 2-4. 사이클 #100 의 세 센서 (1초 평균)
CYCLE = 100
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (sid, label) in zip(axes, [('TS1', '온도'), ('PS1', '압력'), ('FS1', '유량')]):
    ax.plot(range(60), ds.sensors[sid]['mean'][CYCLE])
    ax.set_title(f'{sid} {label} · 사이클 #{CYCLE}')
    ax.set_xlabel('사이클 내 경과 초 (elapsed_s)')
fig.tight_layout()
plt.show()

## 활동 3 · 설비·센서·단위 매핑 표 채우기

위 출력과 `system/data/raw/description.txt` 를 보고 빈칸을 채운다.

| 칸 | 뜻 | 어디서 확인하나 |
|---|---|---|
| `sensor_id` | 그래프(Neo4j)에서 쓰는 센서 ID. **설비ID.센서코드** 모양 | 교재 2절 |
| `unit` | 단위 문자열 | description.txt · 교재 표 |
| `hz` | 1초당 측정 횟수 (정수) | 2-1 출력의 Hz |
| `file` | 원본 파일 이름 | 2-1 에서 연 파일 |

단위는 `'°C'`, `'bar'`, `'L/min'` 처럼 **따옴표 안의 문자열**로 쓴다.

In [ ]:
# 3-1. 매핑 표 — 빈칸(____)을 채운다
ASSET_ID = 'HYD-01'   # 이번 실습에서 UCI 데이터를 재생할 교육용 설비

SENSOR_TABLE = {
    'TS1': {'asset_id': ASSET_ID, 'sensor_id': ____, 'quantity': 'temperature', 'unit': ____, 'hz': ____, 'file': 'TS1.txt'},
    'PS1': {'asset_id': ASSET_ID, 'sensor_id': 'HYD-01.PS1', 'quantity': 'pressure', 'unit': ____, 'hz': ____, 'file': ____},
    'FS1': {'asset_id': ASSET_ID, 'sensor_id': ____, 'quantity': 'flow', 'unit': ____, 'hz': ____, 'file': ____},
    # 경험자: 원본에 있는 다른 센서(예: TS2) 한 줄을 여기에 더해 본다
}

pd.DataFrame(SENSOR_TABLE).T

In [ ]:
# 3-2. 제공 코드(uci.SENSOR_MAP)와 한 칸씩 비교한다
for sid, row in SENSOR_TABLE.items():
    if sid not in uci.SENSOR_MAP:
        print(sid, '제공 코드에 없는 센서 (경험자 확장 칸)')
        continue
    ref = uci.SENSOR_MAP[sid]
    same = {k: row[k] == ref[k] for k in ('file', 'hz', 'unit', 'quantity')}
    print(sid, same)

### 값 하나를 설명하기

아래 셀은 사이클 #100 을 **공통 관측 레코드**로 바꾼 뒤, TS1 의 30초째 값 하나를 보여 준다.
`ts` 는 **재생 시각**(수업에서 정한 시작 시각 + 경과 초)이지 실제 수집 시각이 아니다.

In [ ]:
# 3-3. 공통 관측 레코드 한 줄
REPLAY_START = datetime(2026, 11, 14, 9, 0, tzinfo=timezone.utc)   # 재생 시작 시각 (임의로 정한 값)
rows = uci.cycle_observations(ds, 100, ASSET_ID, REPLAY_START)
print('레코드 수 =', len(rows), '(= 센서 3개 × 60초)')
one = next(r for r in rows if r['sensor_id'] == 'TS1' and r['elapsed_s'] == 30)
one

In [ ]:
# 3-4. 위 값 하나를 내 말로 적는다 — 빈칸을 채운다
ONE_VALUE = {
    'asset_id': ____,
    'sensor_id': 'TS1',
    'origin_cycle_id': 100,
    'elapsed_s': 30,
    'raw_value': ____,          # 소수 셋째 자리까지
    'unit': ____,
    'source_file': ____,
    'is_synthetic': ____,        # UCI 원본이면 False, 시뮬레이터 합성이면 True
}
print(f"설비 {ONE_VALUE['asset_id']} 의 센서 {ONE_VALUE['sensor_id']} 값 {ONE_VALUE['raw_value']} {ONE_VALUE['unit']} 은(는) "
      f"원본 파일 {ONE_VALUE['source_file']} 의 사이클 #{ONE_VALUE['origin_cycle_id']}, {ONE_VALUE['elapsed_s']}초째 값이다.")

## 활동 4 · 재생 버튼으로 그래프 확인하고 저장본 남기기

아래 셀의 ▶ 버튼을 누르면 사이클 세 개를 이어 붙여 **10초씩 재생**한다. `REPLAY_CYCLES` 의 번호를 바꿔 다시 눌러 본다.
원본 CSV 재생은 조치에 반응하지 않는다 — 조치에 반응하는 것은 뒤 회차의 시뮬레이터다.

In [ ]:
# 4-1. 재생
import time
from IPython.display import clear_output, display

REPLAY_CYCLES = [100, 101, 102]
STEP_S, DELAY_S = 10, 0.05

replay_rows = []
for cid, cyc_rows in uci.replay_cycles(ds, REPLAY_CYCLES, ASSET_ID, start=REPLAY_START):
    replay_rows.extend(cyc_rows)
frame = pd.DataFrame(replay_rows).sort_values(['ts', 'sensor_id'])
total_s = len(REPLAY_CYCLES) * 60

for shown in range(STEP_S, total_s + 1, STEP_S):
    part = frame[frame.ts < REPLAY_START + pd.Timedelta(seconds=shown)]
    fig, axes = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True)
    for ax, sid in zip(axes, ['TS1', 'PS1', 'FS1']):
        s = part[part.sensor_id == sid]
        ax.plot((s.ts - REPLAY_START).dt.total_seconds(), s.raw_value)
        ax.set_ylabel(f"{sid} [{SENSOR_TABLE[sid]['unit']}]")
        ax.set_xlim(0, total_s)
    for k in range(1, len(REPLAY_CYCLES)):
        for ax in axes:
            ax.axvline(k * 60, color='gray', linestyle=':')
    axes[0].set_title(f'재생 {shown:3d}/{total_s}초 · 원본 사이클 {REPLAY_CYCLES} · {ASSET_ID}')
    axes[-1].set_xlabel('재생 경과 초 (재생 시각 기준, 실제 수집 시각 아님)')
    fig.tight_layout()
    clear_output(wait=True)
    display(fig)
    plt.close(fig)
    time.sleep(DELAY_S)
print('재생한 관측 레코드 수 =', len(frame))

In [ ]:
# 4-2. B2 미리 보기 — 같은 관측에 품질 플래그를 붙인다 (다음 회차에서 자세히 다룬다)
from hydops.b2_quality.checks import SensorQualityChecker

flags = pd.DataFrame(SensorQualityChecker().check_many(replay_rows))
print(flags.groupby(['sensor_id', 'quality_flag']).size())
flags[flags.quality_flag != 'OK'][['sensor_id', 'origin_cycle_id', 'elapsed_s', 'raw_value', 'value', 'quality_flag']]

In [ ]:
# 4-3. 저장본 남기기 — saved/ 폴더에 CSV 와 마지막 그래프를 남긴다
SAVE_DIR = Path('saved')
SAVE_DIR.mkdir(exist_ok=True)
out = flags.assign(ts=flags.ts.astype(str), agg=flags['agg'].astype(str))
out.to_csv(SAVE_DIR / 'S01_replay_observations.csv', index=False)
fig.savefig(SAVE_DIR / 'S01_replay_last_frame.png', dpi=120)
pd.DataFrame(SENSOR_TABLE).T.to_csv(SAVE_DIR / 'S01_sensor_table.csv')
for p in sorted(SAVE_DIR.glob('S01_*')):
    print(p.name)

In [ ]:
# 5. 자기 점검 — 빈칸이 모두 맞으면 '모든 빈칸 통과' 가 나온다
problems = []
for sid in ('TS1', 'PS1', 'FS1'):
    row, ref = SENSOR_TABLE[sid], uci.SENSOR_MAP[sid]
    for k in ('file', 'hz', 'unit'):
        if row[k] != ref[k]:
            problems.append(f'{sid}.{k}')
    if row['sensor_id'] != f"{row['asset_id']}.{sid}":
        problems.append(f'{sid}.sensor_id')
checks = {
    'asset_id': ONE_VALUE['asset_id'] == one['asset_id'],
    'raw_value': isinstance(ONE_VALUE['raw_value'], (int, float)) and abs(ONE_VALUE['raw_value'] - one['raw_value']) < 0.001,
    'unit': ONE_VALUE['unit'] == one['unit'],
    'source_file': ONE_VALUE['source_file'] == uci.SENSOR_MAP['TS1']['file'],
    'is_synthetic': ONE_VALUE['is_synthetic'] is one['is_synthetic'],
}
problems += [f'ONE_VALUE.{k}' for k, ok in checks.items() if not ok]
print('모든 빈칸 통과' if not problems else '확인 필요: ' + ', '.join(problems))